In [1]:
import json
import random
import re
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from PIL import Image
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ROOT = Path(".").resolve() 
DATASET_PATH = ROOT / "dataset.jsonl"
IMAGE_SIZE = 224
DIM = 256
N_LAYERS = 4
MAX_TEXT_LEN = 128
BATCH_SIZE = 4
EPOCHS = 8
LR = 3e-4
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
print("device:", DEVICE)
print("dataset:", DATASET_PATH)

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'dlopen(/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so, 0x0006): Symbol not found: (__ZN3c1017RegisterOperatorsD1Ev)
  Referenced from: '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torchvision/image.so'
  Expected in: '/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/lib/libtorch_cpu.dylib''If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipyw

device: cpu
dataset: /Users/nikitakamenev/Documents/scince/ITMO/designing_neural_network_architectures_2025_02/seminar_05/dataset.jsonl


### LLM

In [2]:
class SelfAttention(nn.Module):
    def __init__(self, dim, causal=False):
        super().__init__()
        self.query = nn.Linear(dim, dim)
        self.key = nn.Linear(dim, dim)
        self.value = nn.Linear(dim, dim)
        self.scale = dim ** 0.5
        self.causal = causal

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)

        scores = Q @ K.transpose(-2, -1) / self.scale # K_t = K.permute(0, 2, 1)

        if self.causal:
            seq_len = x.size(1)
            mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device, dtype=torch.bool))
            scores = scores.masked_fill(~mask, float("-inf"))

        weights = torch.softmax(scores, dim=-1)
        return weights @ V


class TransformerBlock(nn.Module):
    def __init__(self, dim, causal=False):
        super().__init__()
        self.attention = SelfAttention(dim, causal=causal)
        self.norm1 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim),
        )
        self.norm2 = nn.LayerNorm(dim)

    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x


class MiniLLM(nn.Module):

    def __init__(self, vocab_size, dim, n_layers):
        super().__init__()
        self.blocks = nn.ModuleList(
            [TransformerBlock(dim, causal=True) for _ in range(n_layers)]
        )
        self.norm = nn.LayerNorm(dim)
        self.lm_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, x):
        for block in self.blocks:
            x = block(x)
        x = self.norm(x)
        return self.lm_head(x)

### ViT-энкодер



In [3]:
class VisionEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()
        weights = ViT_B_16_Weights.IMAGENET1K_V1
        self.vit = vit_b_16(weights=weights)
        self.vit.heads = nn.Identity()
        for p in self.vit.parameters():
            p.requires_grad = False
        self.proj = nn.Linear(768, out_dim)

    def forward(self, images):
        # patch tokens без [CLS]
        x = self.vit._process_input(images)
        batch_size = x.shape[0]
        cls = self.vit.class_token.expand(batch_size, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = self.vit.encoder(x)
        patch_tokens = x[:, 1:, :]
        return self.proj(patch_tokens)

In [4]:
class VisionEncoder(nn.Module):
    def __init__(self, out_dim):
        super().__init__()

        # берем готовый ViT
        self.vit = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)

        # убираем финальную классификационную голову
        self.vit.heads = nn.Identity()

        # замораживаем ViT (обучаем только проекцию ниже)
        for p in self.vit.parameters():
            p.requires_grad = False

        # приводим размерность признаков ViT (768) к out_dim
        self.proj = nn.Linear(768, out_dim)

    def forward(self, images):

        # ViT возвращает один вектор на изображение: (B, 768)
        img_features = self.vit(images)

        # Добавляем чтобы потом можно было cat с text токенами
        img_features = img_features.unsqueeze(1)  # (B, 1, 768)

        # Проекция в размерность модели
        return self.proj(img_features)  # (B, 1, out_dim)

## Простой токенизатор по словам



In [5]:
class WordTokenizer:
    PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"
    SPECIAL = [PAD, BOS, EOS, UNK]

    def __init__(self, texts):
        tokens = []
        for t in texts:
            tokens.extend(self._tokenize(t))

        vocab_tokens = sorted(set(tokens))
        self.itos = self.SPECIAL + vocab_tokens
        self.stoi = {tok: i for i, tok in enumerate(self.itos)}
        self.pad_id = self.stoi[self.PAD]
        self.bos_id = self.stoi[self.BOS]
        self.eos_id = self.stoi[self.EOS]
        self.unk_id = self.stoi[self.UNK]

    @staticmethod
    def _tokenize(text):
        return re.findall(r"\w+|[^\w\s]", text, flags=re.UNICODE)

    def __len__(self):
        return len(self.itos)

    def encode(self, text, max_len):
        toks = self._tokenize(text)
        ids = [self.bos_id] + [self.stoi.get(tok, self.unk_id) for tok in toks] + [self.eos_id]
        ids = ids[:max_len]
        ids += [self.pad_id] * (max_len - len(ids))
        return ids

    def decode(self, ids):
        pieces = []
        for i in ids:
            tok = self.itos[i]
            if tok == self.EOS:
                break
            if tok not in self.SPECIAL:
                pieces.append(tok)

        text = " ".join(pieces)
        text = re.sub(r"\s+([.,!?;:])", r"\1", text)
        return text

In [6]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def resolve_image_path(image_field: str) -> Path:
    p = Path(image_field.replace("\\", "/"))
    if p.is_absolute():
        return p
    return (ROOT / p).resolve()


class CaptionDataset(Dataset):
    def __init__(self, rows, tokenizer, transform):
        self.rows = []
        for row in rows:
            img_path = resolve_image_path(row["image"])
            if img_path.exists():
                self.rows.append({"image": img_path, "text": row["text"]})
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, idx):
        item = self.rows[idx]
        image = Image.open(item["image"]).convert("RGB")
        image = self.transform(image)
        text_ids = torch.tensor(
            self.tokenizer.encode(item["text"], MAX_TEXT_LEN), dtype=torch.long
        )
        return image, text_ids


transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

all_rows = load_jsonl(DATASET_PATH)
tokenizer = WordTokenizer([r["text"] for r in all_rows])
full_ds = CaptionDataset(all_rows, tokenizer, transform)

n_val = max(1, len(full_ds) // 10)
n_train = len(full_ds) - n_val
train_ds, val_ds = random_split(
    full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(SEED)
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f"samples: train={len(train_ds)}, val={len(val_ds)}, vocab={len(tokenizer)}")

samples: train=157, val=17, vocab=1246


## SmallVLM

In [7]:
class SmallVLM(nn.Module):


    def __init__(self, vocab_size, dim, n_layers):
        super().__init__()
        self.vision_encoder = VisionEncoder(dim)      
        self.text_embedding = nn.Embedding(vocab_size, dim) 
        self.language_model = MiniLLM(vocab_size, dim, n_layers)

    def forward(self, images, text_ids):

        # визуальные токены
        vision_tokens = self.vision_encoder(images)   # (B, N_vis, D)
        n_vis = vision_tokens.size(1)

        text_input_ids = text_ids[:, :-1]
        text_tokens = self.text_embedding(text_input_ids)

        # склеиваем
        joint_tokens = torch.cat([vision_tokens, text_tokens], dim=1)

        all_logits = self.language_model(joint_tokens)  # (B, N_vis+T-1, vocab)

        # берем только ту часть, которая соответствует тексту
        text_logits = all_logits[:, n_vis:, :]
        return text_logits

    @torch.no_grad()
    def generate(self, image, tokenizer, max_new_tokens=80):

        self.eval()

        image = image.unsqueeze(0).to(DEVICE)

        vision_tokens = self.vision_encoder(image)

        # начало последовательности
        generated_ids = torch.tensor([[tokenizer.bos_id]], device=DEVICE)

        for _ in range(max_new_tokens):
            text_tokens = self.text_embedding(generated_ids)
            joint_tokens = torch.cat([vision_tokens, text_tokens], dim=1)

            logits = self.language_model(joint_tokens) 
            next_token_id = logits[:, -1, :].argmax(dim=-1, keepdim=True)

            generated_ids = torch.cat([generated_ids, next_token_id], dim=1)

            if next_token_id.item() == tokenizer.eos_id:
                break

        return tokenizer.decode(generated_ids[0].tolist())

## Обучение

In [8]:
def compute_loss(logits, text_ids, pad_id):
    targets = text_ids[:, 1:].contiguous()
    return F.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        targets.reshape(-1),
        ignore_index=pad_id,
    )


model = SmallVLM(len(tokenizer), DIM, N_LAYERS).to(DEVICE)
optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad], lr=LR
)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss, n_batches = 0.0, 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for images, text_ids in tqdm(loader, leave=False):
            images = images.to(DEVICE)
            text_ids = text_ids.to(DEVICE)
            logits = model(images, text_ids)
            loss = compute_loss(logits, text_ids, tokenizer.pad_id)
            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            total_loss += loss.item()
            n_batches += 1
    return total_loss / max(n_batches, 1)


for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, train=True)
    val_loss = run_epoch(val_loader, train=False)
    print(f"epoch {epoch:02d} | train loss {train_loss:.4f} | val loss {val_loss:.4f}")

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /Users/nikitakamenev/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth
100%|██████████| 330M/330M [00:48<00:00, 7.21MB/s] 


epoch 01 | train loss 5.2989 | val loss 4.3660


epoch 02 | train loss 3.6873 | val loss 3.7822


epoch 03 | train loss 3.0574 | val loss 3.6204


epoch 04 | train loss 2.6664 | val loss 3.5591


epoch 05 | train loss 2.3583 | val loss 3.4830


epoch 06 | train loss 2.0802 | val loss 3.4691


epoch 07 | train loss 1.8644 | val loss 3.4365


epoch 08 | train loss 1.6544 | val loss 3.4726


In [ ]:

sample_idx = 0
image, text_ids = full_ds[sample_idx]
gt_text = tokenizer.decode(text_ids.tolist())
pred_text = model.generate(image, tokenizer)

print("GT:", gt_text[:300])

print("Pred:", pred_text[:300])


GT: В изображении представлено просторное помещение, возможно, это офис или учебный класс. На переднем плане видны несколько стеллажей, на которых хранятся различные предметы, включая коробки. В центре изображения находится стол, на котором лежит несколько папок. В левом углу можно увидеть человека, кот
Pred: На изображении представлено помещение, возможно, это лаборатория или офис. В центре изображения видны несколько стульев, на которых хранятся различные предметы, включая пакеты и оборудование. В центре изображения можно увидеть несколько предметов, включая пакеты и оборудование.


In [ ]:
ckpt_path = "small_vlm.pt"
torch.save({
    "model": model.state_dict(),
    "tokenizer_itos": tokenizer.itos,
    "config": {"dim": DIM, "n_layers": N_LAYERS, "max_text_len": MAX_TEXT_LEN},
}, ckpt_path)
print("saved:", ckpt_path)